# 01 — Persiapan & Integrasi Data
**Proyek UAS Visualisasi Data dan Informasi 2026 — "Terang, Kaya, Sejahtera?"**
Nabhan Athallah (3SD2 / 222313272)

Notebook ini menggabungkan seluruh data yang sudah divalidasi menjadi tabel siap pakai untuk analisis (notebook 02) dan dashboard Streamlit.

| Sumber | Isi | File input |
|---|---|---|
| BPS | PDRB per kapita, laju PDRB, IPM, P0–P2, IKK, penduduk, kepadatan (514 kab/kota, 2024) | `master_kabkota_2024.csv` |
| BPS | Indikator 38 provinsi 2024 | `master_provinsi_2024.csv` |
| BPS | PDRB ADHB menurut 17 lapangan usaha provinsi 2024 | `pdrb_lapangan_usaha_provinsi_2024.csv` |
| BPS (LF SP2020) | Migrasi risen antarprovinsi (Tabel 5.3 & 7) | `migrasi_risen_*.csv` |
| VIIRS VNL (GEE) | Statistik zonal cahaya malam 2020 & 2024 | `viirs_kabkota_2020_2024.csv` |
| geoBoundaries (BPS–WFP–OCHA) | Batas 514 kab/kota & 38 provinsi | `*.geojson` |

In [1]:
import json, numpy as np, pandas as pd, geopandas as gpd
from pathlib import Path
pd.set_option("display.max_columns", 50)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IN, OUT = ROOT / "data" / "input", ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
print(ROOT)

/home/claude/repo


## 1. Membaca data BPS kab/kota dan hasil GEE

In [2]:
kab = pd.read_csv(IN / "master_kabkota_2024.csv", dtype={"kode_bps": str, "kode_prov": str, "kode_kemendagri": str})
prov = pd.read_csv(IN / "master_provinsi_2024.csv", dtype={"kode_prov": str})
viirs = pd.read_csv(IN / "viirs_kabkota_2020_2024.csv", dtype={"kode_bps": str})
print(kab.shape, prov.shape, viirs.shape)
assert kab.kode_bps.is_unique and len(kab) == 514
assert len(viirs) == 1028 and viirs.isna().sum().sum() == 0
assert set(viirs.kode_bps) == set(kab.kode_bps)
viirs.groupby(["tahun", "versi_vnl"]).size()

(514, 20) (38, 9) (1028, 10)


tahun  versi_vnl
2020   V2.1         514
2024   V2.2         514
dtype: int64

## 2. Pembersihan VIIRS
* Nilai negatif sangat kecil (≥ −3, akibat koreksi latar belakang) diubah menjadi 0.
* Ubah ke format lebar (satu baris per kab/kota).

In [3]:
print("nilai rad_sum negatif:", (viirs.rad_sum < 0).sum())
for c in ["rad_sum", "rad_mean"]:
    viirs[c] = viirs[c].clip(lower=0)
w = viirs.pivot(index="kode_bps", columns="tahun",
                values=["rad_sum", "rad_mean", "rad_max", "lit_share", "cfcvg_mean", "n_pix"])
w.columns = [f"{a}_{b}" for a, b in w.columns]
w = w.reset_index()
w.head()

nilai rad_sum negatif: 1


,kode_bps,rad_sum_2020,rad_sum_2024,rad_mean_2020,rad_mean_2024,rad_max_2020,rad_max_2024,lit_share_2020,lit_share_2024,cfcvg_mean_2020,cfcvg_mean_2024,n_pix_2020,n_pix_2024
0,1101,209.105894,241.520369,0.024301,0.028068,7.723139,9.286028,0.013942,0.015519,28.840409,16.051275,9385.0,9385.0
1,1102,716.145079,691.113206,0.081065,0.078231,9.258633,10.334950,0.047302,0.046176,26.852489,16.111536,9486.0,9486.0
2,1103,429.964975,852.329380,0.021760,0.043136,8.942591,9.596172,0.015172,0.027227,23.879921,15.663001,20365.0,20365.0
3,1104,428.383854,667.634112,0.022041,0.034351,15.266407,13.192988,0.011888,0.014923,26.327953,21.109231,19911.0,19911.0
4,1105,4667.858508,4994.398193,0.182799,0.195587,324.204224,232.054276,0.070753,0.075351,24.717395,31.328500,26175.0,26175.0


## 3. Variabel turunan lensa cahaya
| Variabel | Rumus |
|---|---|
| `cahaya_pk` | Σ radiance 2024 ÷ penduduk 2024 × 1.000 (nW/cm²/sr per 1.000 jiwa) |
| `pct_menyala` | proporsi piksel > 0,5 nW/cm²/sr × 100 |
| `pertumbuhan_cahaya` | 100 × [ln(Σrad₂₀₂₄ + 1) − ln(Σrad₂₀₂₀ + 1)] (≈ % perubahan; stabil untuk nilai mendekati 0) |
| `cahaya_rata` | rata-rata radiance per piksel 2024 (intensitas per luas) |
| `flag_awan` | `True` bila rata-rata malam bebas awan 2024 < 15 (mutu rendah) |

In [4]:
df = kab.merge(w, on="kode_bps", how="left", validate="1:1")
df["cahaya_pk"] = df.rad_sum_2024 / df.penduduk_2024 * 1000
df["pct_menyala"] = df.lit_share_2024 * 100
df["pertumbuhan_cahaya"] = 100 * (np.log(df.rad_sum_2024 + 1) - np.log(df.rad_sum_2020 + 1))
df["cahaya_rata"] = df.rad_mean_2024
df["flag_awan"] = df.cfcvg_mean_2024 < 15
df[["cahaya_pk", "pct_menyala", "pertumbuhan_cahaya", "cahaya_rata"]].describe().round(3)

,cahaya_pk,pct_menyala,pertumbuhan_cahaya,cahaya_rata
count,514.000,514.000,514.000,514.000
mean,9.655,32.309,41.975,2.505
std,10.394,35.452,38.141,5.924
min,0.000,0.000,-47.986,0.000
25%,3.873,3.647,21.901,0.061
50%,7.626,12.277,37.029,0.260
75%,11.316,63.046,53.797,1.672
max,108.678,100.000,350.689,52.791


In [5]:
# Cek kewajaran: paling terang & paling gelap (per luas)
cols = ["nama", "provinsi", "cahaya_rata", "cahaya_pk", "pct_menyala", "cfcvg_mean_2024"]
display(df.nlargest(5, "cahaya_rata")[cols]); display(df.nsmallest(5, "cahaya_rata")[cols])
print("kab/kota mutu awan rendah:", df.flag_awan.sum())

,nama,provinsi,cahaya_rata,cahaya_pk,pct_menyala,cfcvg_mean_2024
157,Kota Jakarta Pusat,DKI Jakarta,52.791077,11.380694,100.0,37.803268
159,Kota Jakarta Utara,DKI Jakarta,36.684565,13.353772,100.0,37.318046
158,Kota Jakarta Barat,DKI Jakarta,35.809494,8.513306,100.0,40.121873
155,Kota Jakarta Selatan,DKI Jakarta,34.451018,10.543292,100.0,47.006595
263,Kota Surabaya,Jawa Timur,29.561748,16.030915,100.0,65.891334


,nama,provinsi,cahaya_rata,cahaya_pk,pct_menyala,cfcvg_mean_2024
504,Kabupaten Puncak,Papua Tengah,0.000000,0.000000,0.000000,9.093630
506,Kabupaten Nduga,Papua Pegunungan,0.000000,0.000000,0.000000,6.746145
510,Kabupaten Mamberamo Tengah,Papua Pegunungan,0.000000,0.000000,0.000000,12.751913
513,Kabupaten Pegunungan Bintang,Papua Pegunungan,0.000000,0.000000,0.000000,8.333123
492,Kabupaten Mamberamo Raya,Papua,0.000103,0.354018,0.009603,20.013917


kab/kota mutu awan rendah: 47


## 4. Agregasi lensa cahaya ke 38 provinsi (penjumlahan total radiance)

In [6]:
g = df.groupby("kode_prov").agg(rad_sum_2024=("rad_sum_2024", "sum"), rad_sum_2020=("rad_sum_2020", "sum"),
                                 lit_px=("lit_share_2024", lambda s: (s * df.loc[s.index, "n_pix_2024"]).sum()),
                                 n_pix=("n_pix_2024", "sum"), jumlah_kabkota=("kode_bps", "count")).reset_index()
pv = prov.merge(g, on="kode_prov", how="left", validate="1:1")
pv["penduduk_2024"] = pv.penduduk_ribu * 1000
pv["cahaya_pk"] = pv.rad_sum_2024 / pv.penduduk_2024 * 1000
pv["pct_menyala"] = pv.lit_px / pv.n_pix * 100
pv["pertumbuhan_cahaya"] = 100 * (np.log(pv.rad_sum_2024 + 1) - np.log(pv.rad_sum_2020 + 1))
# laju PDRB provinsi tidak tersedia di tabel provinsi → rata-rata tertimbang PDRB kab/kota
lw = df.assign(b=df.pdrb_adhb_miliar).groupby("kode_prov").apply(lambda x: np.average(x.laju_pdrb_pct, weights=x.b), include_groups=False)
pv["laju_pdrb_pct"] = pv.kode_prov.map(lw)
pv = pv.drop(columns=["lit_px", "n_pix"])
assert pv.isna().sum().sum() == 0
pv.head()

,kode_prov,provinsi,pulau,ikk,pdrb_kapita_ribu,penduduk_ribu,kepadatan,ipm,p0,rad_sum_2024,rad_sum_2020,jumlah_kabkota,penduduk_2024,cahaya_pk,pct_menyala,pertumbuhan_cahaya,laju_pdrb_pct
0,11,Aceh,Sumatera,96.61,43782.0,5554.8,98,74.03,14.23,57034.308098,44952.549095,23,5554800.0,10.267572,6.585940,23.804080,4.596022
1,12,Sumatera Utara,Sumatera,97.45,73575.0,15588.5,215,74.02,7.99,142457.948597,110327.586837,33,15588500.0,9.138657,10.458952,25.559081,4.888119
2,13,Sumatera Barat,Sumatera,93.06,57083.0,5836.2,139,74.49,5.97,32335.494546,24084.483088,19,5836200.0,5.540505,6.248759,29.458716,4.256000
3,14,Riau,Sumatera,96.10,165350.0,6728.1,75,74.79,6.67,103079.005458,88640.369297,12,6728100.0,15.320671,8.068983,15.090677,3.158991
4,15,Jambi,Sumatera,95.32,86775.0,3724.3,76,73.43,7.10,52717.703302,37265.461328,11,3724300.0,14.155064,6.939460,34.687653,4.491283


## 5. Geometri ringan untuk dashboard (presisi koordinat 4 desimal ≈ 11 m)

In [7]:
from shapely import set_precision
def ringan(path, keep):
    g = gpd.read_file(path)[keep + ["geometry"]]
    g["geometry"] = set_precision(g.geometry.values, 1e-4)
    g = g[~g.geometry.is_empty]
    return g
gk = ringan(IN / "kabkota_514_dashboard.geojson", ["kode_bps"])
gp = ringan(IN / "provinsi_38_dashboard.geojson", ["kode_prov"])
gk["kode_bps"] = gk.kode_bps.astype(str); gp["kode_prov"] = gp.kode_prov.astype(str)
assert len(gk) == 514 and len(gp) == 38 and set(gk.kode_bps) == set(df.kode_bps)
# titik representatif (di dalam poligon) untuk peta simbol proporsional & label
rp = gk.set_index("kode_bps").to_crs(3395).representative_point().to_crs(4326)
df["lon"], df["lat"] = df.kode_bps.map(rp.x).round(4), df.kode_bps.map(rp.y).round(4)
rpp = gp.set_index("kode_prov").to_crs(3395).representative_point().to_crs(4326)
pv["lon"], pv["lat"] = pv.kode_prov.map(rpp.x).round(4), pv.kode_prov.map(rpp.y).round(4)
gk.to_file(OUT / "kabkota.geojson", driver="GeoJSON", COORDINATE_PRECISION=4)
gp.to_file(OUT / "provinsi.geojson", driver="GeoJSON", COORDINATE_PRECISION=4)
for f in ["kabkota.geojson", "provinsi.geojson"]:
    print(f, round((OUT / f).stat().st_size / 1e6, 2), "MB")

kabkota.geojson 2.13 MB
provinsi.geojson 1.5 MB


## 6. Data hierarki (sektor) dan aliran (migrasi)

In [8]:
sek = pd.read_csv(IN / "pdrb_lapangan_usaha_provinsi_2024.csv", dtype={"kode_prov": str})
od = pd.read_csv(IN / "migrasi_risen_od_2020.csv")
neto = pd.read_csv(IN / "migrasi_risen_neto_provinsi_2020.csv")
mat = pd.read_csv(IN / "migrasi_risen_matriks_34x34_2020.csv", index_col=0)
PULAU34 = {"Aceh": "Sumatera", "Sumatera Utara": "Sumatera", "Sumatera Barat": "Sumatera", "Riau": "Sumatera", "Jambi": "Sumatera",
           "Sumatera Selatan": "Sumatera", "Bengkulu": "Sumatera", "Lampung": "Sumatera", "Kep. Bangka Belitung": "Sumatera",
           "Kepulauan Riau": "Sumatera", "DKI Jakarta": "Jawa", "Jawa Barat": "Jawa", "Jawa Tengah": "Jawa", "DI Yogyakarta": "Jawa",
           "Jawa Timur": "Jawa", "Banten": "Jawa", "Bali": "Bali-Nusa Tenggara", "Nusa Tenggara Barat": "Bali-Nusa Tenggara",
           "Nusa Tenggara Timur": "Bali-Nusa Tenggara", "Kalimantan Barat": "Kalimantan", "Kalimantan Tengah": "Kalimantan",
           "Kalimantan Selatan": "Kalimantan", "Kalimantan Timur": "Kalimantan", "Kalimantan Utara": "Kalimantan",
           "Sulawesi Utara": "Sulawesi", "Sulawesi Tengah": "Sulawesi", "Sulawesi Selatan": "Sulawesi", "Sulawesi Tenggara": "Sulawesi",
           "Gorontalo": "Sulawesi", "Sulawesi Barat": "Sulawesi", "Maluku": "Maluku", "Maluku Utara": "Maluku",
           "Papua Barat": "Papua", "Papua": "Papua"}
od["pulau_asal"] = od.provinsi_asal.map(PULAU34); od["pulau_tujuan"] = od.provinsi_tujuan.map(PULAU34)
neto["pulau"] = neto.provinsi.map(PULAU34)
assert od.pulau_asal.notna().all() and neto.pulau.notna().all()
# validasi: total migran OD = total keluar Tabel 7 (selisih pembulatan)
print("Σ OD:", od.jumlah_migran.sum(), "| Σ keluar Tabel 7:", neto.keluar_risen.sum())
print("Σ 17 kategori vs total PDRB provinsi (maks selisih %):",
      ((sek.drop_duplicates(["kode_prov", "kode_kategori"]).groupby("kode_prov").adhb_kategori.sum()
        / sek.groupby("kode_prov").adhb_2024_miliar.sum()) - 1).abs().max() * 100)

Σ OD: 4167850 | Σ keluar Tabel 7: 4167853
Σ 17 kategori vs total PDRB provinsi (maks selisih %): 0.007980845969668415


## 7. Simpan data terolah

In [9]:
df.to_parquet(OUT / "kabkota.parquet", index=False); df.to_csv(OUT / "kabkota.csv", index=False)
pv.to_parquet(OUT / "provinsi.parquet", index=False); pv.to_csv(OUT / "provinsi.csv", index=False)
sek.to_parquet(OUT / "sektor_provinsi.parquet", index=False)
od.to_parquet(OUT / "migrasi_od.parquet", index=False)
neto.to_parquet(OUT / "migrasi_neto.parquet", index=False)
mat.to_parquet(OUT / "migrasi_matriks.parquet")
print(sorted(p.name for p in OUT.iterdir()))

['kabkota.csv', 'kabkota.geojson', 'kabkota.parquet', 'kabkota_analisis.csv', 'kabkota_analisis.parquet', 'migrasi_matriks.parquet', 'migrasi_neto.parquet', 'migrasi_od.parquet', 'profil_klaster.csv', 'provinsi.csv', 'provinsi.geojson', 'provinsi.parquet', 'ringkasan_analisis.json', 'sektor_provinsi.parquet']
